# 10 — Neural Inertial Odometry Held-Out Test Evaluation

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 18:** Testing notebook must load the saved checkpoint and evaluate held-out data.
> **Section 9:** Held-out testing on unseen Driver A session `S1`.

### Objectives:
1. Load trained model from `checkpoints/inertial_odometry/inertial_odometry_best.pt`.
2. Evaluate on unseen held-out driving session `S1` (Driver A).
3. Benchmark forward velocity tracking against CAN-bus vehicle speed reference.
4. Benchmark relative displacement estimation and predictive uncertainty calibration.

## 1. Environment & Checkpoint Loading

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.inertial_odometry import NeuralInertialOdometry
from src.datasets.inertial_odometry_dataset import InertialOdometryDataset
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'
plots_dir = PROJECT_ROOT / 'plots' / 'inertial_odometry'
plots_dir.mkdir(parents=True, exist_ok=True)

print(f'Loading Neural Inertial Odometry checkpoint: {ckpt_path}')
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found at {ckpt_path}. Run 09_inertial_odometry_training.ipynb first.')

ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt.get('config', {})

model = NeuralInertialOdometry(
    input_dim=6,
    tcn_channels=cfg.get('tcn_channels', [64, 128, 256]),
    kernel_size=cfg.get('tcn_kernel_size', 3)
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Model loaded from Epoch {ckpt.get("epoch")} (Best Val Loss: {ckpt.get("best_val_loss"):.4f})')

## 2. Load Held-Out Test Motion Dataset (Driver A — Session S1)

In [ ]:
test_ds = InertialOdometryDataset(
    split='test',
    window_size=100,
    stride=20
)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(f'Held-Out Test Windows: {len(test_ds)}')

## 3. Quantitative Evaluation on Held-Out Data

In [ ]:
all_pred_disp = []
all_gt_disp = []
all_pred_vel = []
all_gt_vel = []
all_pred_sigma = []

with torch.no_grad():
    for batch in test_loader:
        x_imu = batch['imu'].to(device)
        gt_d = batch['disp'].to(device)
        gt_v = batch['vel'].to(device)
        
        p_disp, p_vel, p_logvar = model(x_imu)
        p_sigma = torch.exp(0.5 * p_logvar)
        
        all_pred_disp.append(p_disp.cpu().numpy())
        all_gt_disp.append(gt_d.cpu().numpy())
        all_pred_vel.append(p_vel.cpu().numpy())
        all_gt_vel.append(gt_v.cpu().numpy())
        all_pred_sigma.append(p_sigma.cpu().numpy())

pred_disp = np.vstack(all_pred_disp)
gt_disp = np.vstack(all_gt_disp)
pred_vel = np.vstack(all_pred_vel)
gt_vel = np.vstack(all_gt_vel)
pred_sigma = np.vstack(all_pred_sigma)

# Metrics
disp_errors = np.linalg.norm(pred_disp - gt_disp, axis=1)
disp_rmse = float(np.sqrt(np.mean(disp_errors ** 2)))
disp_mae = float(np.mean(disp_errors))

vel_fwd_errors = np.abs(pred_vel[:, 0] - gt_vel[:, 0])
vel_rmse = float(np.sqrt(np.mean(vel_fwd_errors ** 2)))
vel_corr = float(np.corrcoef(pred_vel[:, 0], gt_vel[:, 0])[0, 1])

print('=' * 65)
print('NEURAL INERTIAL ODOMETRY — HELD-OUT TEST RESULTS (SESSION S1)')
print('=' * 65)
print(f'Displacement RMSE (10s window) : {disp_rmse:.3f} meters')
print(f'Displacement MAE               : {disp_mae:.3f} meters')
print(f'Forward Velocity RMSE          : {vel_rmse:.3f} m/s ({vel_rmse * 3.6:.2f} km/h)')
print(f'Velocity Correlation with CAN  : r = {vel_corr:.4f}')
print(f'Mean Predicted Displacement Sigma: {np.mean(pred_sigma):.3f} meters')
print('=' * 65)

## 4. Visual Diagnostics: Velocity Tracking & Displacement Estimation

In [ ]:
# Plot 1: Forward Velocity Tracking vs CAN Ground Truth
n_pts = min(250, len(pred_vel))
t_axis = np.arange(n_pts) * 2.0  # 2s stride

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_axis, gt_vel[:n_pts, 0] * 3.6, 'k-', linewidth=2.0, label='CAN-Bus Reference Speed', alpha=0.8)
ax.plot(t_axis, pred_vel[:n_pts, 0] * 3.6, color='crimson', linestyle='--', linewidth=1.8, label=f'Neural Odometry Velocity (RMSE: {vel_rmse*3.6:.1f} km/h, r: {vel_corr:.2f})')
ax.fill_between(t_axis, (pred_vel[:n_pts, 0] - pred_sigma[:n_pts, 0]) * 3.6, (pred_vel[:n_pts, 0] + pred_sigma[:n_pts, 0]) * 3.6, color='crimson', alpha=0.15, label='Predictive Uncertainty (±1σ)')

ax.set_xlabel('Driving Elapsed Time [s]', fontweight='bold')
ax.set_ylabel('Vehicle Forward Speed [km/h]', fontweight='bold')
ax.set_title('Neural Inertial Odometry: Learned Forward Velocity Tracking on Held-Out Session S1', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
p1_path = plots_dir / 'test_velocity_tracking_S1.png'
plt.savefig(p1_path, dpi=200)
plt.close()
print(f'Velocity tracking plot saved: {p1_path}')

# Plot 2: Relative Displacement Scatter & Error CDF
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(gt_disp[:, 0], pred_disp[:, 0], alpha=0.4, color='royalblue', edgecolors='none', s=25)
lims = [min(np.min(gt_disp[:, 0]), np.min(pred_disp[:, 0])), max(np.max(gt_disp[:, 0]), np.max(pred_disp[:, 0]))]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Agreement (y = x)')
axes[0].set_xlabel('True Forward Displacement [m]', fontweight='bold')
axes[0].set_ylabel('Predicted Forward Displacement [m]', fontweight='bold')
axes[0].set_title(f'Displacement Estimation (RMSE: {disp_rmse:.2f}m)', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.4)

# Error CDF
sorted_errs = np.sort(disp_errors)
cdf = np.arange(1, len(sorted_errs) + 1) / len(sorted_errs)
p95 = np.percentile(sorted_errs, 95)
axes[1].plot(sorted_errs, cdf, color='forestgreen', linewidth=2.0, label='Error CDF')
axes[1].axvline(p95, color='crimson', linestyle=':', label=f'95th Percentile: {p95:.2f}m')
axes[1].set_xlabel('Displacement Error [m]', fontweight='bold')
axes[1].set_ylabel('Cumulative Probability', fontweight='bold')
axes[1].set_title('Displacement Error Cumulative Distribution (CDF)', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
p2_path = plots_dir / 'test_displacement_error_S1.png'
plt.savefig(p2_path, dpi=200)
plt.close()
print(f'Displacement evaluation plot saved: {p2_path}')

# Save Results
test_metrics = {
    'session': 'S1',
    'split': 'held_out_test',
    'displacement_rmse_m': round(disp_rmse, 3),
    'displacement_mae_m': round(disp_mae, 3),
    'displacement_p95_m': round(float(p95), 3),
    'velocity_rmse_mps': round(vel_rmse, 3),
    'velocity_rmse_kmh': round(vel_rmse * 3.6, 2),
    'velocity_correlation_with_can': round(vel_corr, 4),
    'mean_predicted_uncertainty_sigma': round(float(np.mean(pred_sigma)), 3),
    'status': 'VERIFIED'
}

out_res = PROJECT_ROOT / 'results' / 'inertial_odometry_results.json'
with open(out_res, 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'Test metrics saved to {out_res}')